# pull together different data - understand where aircraft is wrt cloud top

code to make a csv estimating distance from cloud top of the forward facing camera (ffc) and rear facing camera (rfc)

In [1]:
import sys
import math
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
from netCDF4 import Dataset # !! new - building netcdf
import os # file saving 
import xarray as xr
from pathlib import Path
import importlib.util
from glob import glob
import seaborn as sns
import imageio 
import tempfile
import pandas as pd


sys.path.append('/home/users/esree/dcmex_instruments/faam_data/')
# loads of homemade functions <3 <3
from faam_data_functions import faam_mo_files, time_adjust, get_nc_variable, mask_data, flatten_20hz_variable, mo_mean, datetime_to_sec, cloud_pass_time
from faam_data_functions_v2 import faam_mo_files_v2, make_cdp_psd_v2, make_cdp_conc, cloud_pass_time_v2



## ffc / rfc - H. Burns work
want to get mean of the two values which approximate distance aicraft is from cloud top
<br>
there are 2 distances calculated, will do mean of them seperately, them mean of both

In [2]:
mmdd = '0730'
fc_date = f'2022{mmdd}'
date = f'22{mmdd}'
fc_loc = f'/gws/ssde/j25a/dcmex/users/hburns/DCMEX2/cloud_heights/{fc_date}/'
ffc_file = f'cloud_heights_{fc_date}_ffc.csv'
rfc_file = f'cloud_heights_{fc_date}_rfc.csv'

csv_save_loc = '/home/users/esree/simple-graupel-model/tune_to_DCMEX/other_numbers/'

In [3]:
ffc_df = pd.read_csv(fc_loc + ffc_file)
rfc_df = pd.read_csv(fc_loc + rfc_file)
## meaning of columns? - which have numbers
## 2 measurements are taken of cloud top approximation

# d1 / d2 = distance from cloud (m) 
# plane_height = altitude of plane (m) (same as ALT_GIN from faam)
# cloud_height1/2 = cloud top height (m) (can be used to compare to plane height)
# roll = aircraft banking left / right (wings moving down / up)
# pitch = aircraft ascending / decending (nose pointing up / down)
# x_target = ??
# pass_number = seperating different passes - but has not done best job
# camera = ffc / rfc - which aircraft camera
# timestamp = timestamp of measurement (UTC)
# start/end_pass_height = fixed values for aircraft start and end of pass alt (m)

## VVV i think this value is wrong VVV - doesn't match the other given values??
#### cloud_above_plane_height = difference between cloud height and plane height (m)
# pixel_height = something to do with camera?


# mask out values where the cloud height is less than the plane height - too far away and underestimate

ffc_df

,Unnamed: 0,d1,d2,plane_height,pass_height,cloud_height1,cloud_height2,cloud above plane height1,cloud above plane height2,roll,pitch,pixel height,x_target,pass_number,camera,timestamp,start_pass_height,end_pass_height,cloud_above_plane_height,pixel_height
0,0,5966.7640,6560.9990,5655.8680,NaN,6436.2773,6513.1006,NaN,NaN,-6.389091,2.883346,NaN,69,271,ffc,2022-07-30 16:29:07,5419.031,5394.9270,1118.17380,496.0
1,1,5817.8086,6412.1140,5649.4260,NaN,6055.6675,6096.2620,NaN,NaN,-6.267916,2.896390,NaN,664,271,ffc,2022-07-30 16:29:08,5419.031,5394.9270,701.33545,401.0
2,2,5668.7670,6263.1400,5643.0757,NaN,5951.5960,5983.0664,NaN,NaN,-6.035450,2.921747,NaN,699,271,ffc,2022-07-30 16:29:09,5419.031,5394.9270,588.13965,377.0
3,3,5519.1960,6113.6294,5636.8057,NaN,6178.0400,6235.4870,NaN,NaN,-5.697225,2.927550,NaN,701,271,ffc,2022-07-30 16:29:10,5419.031,5394.9270,840.56006,445.0
4,4,5370.3037,5964.7960,5630.6330,NaN,6140.0347,6195.5970,NaN,NaN,-5.413175,2.900872,NaN,699,271,ffc,2022-07-30 16:29:11,5419.031,5394.9270,800.67040,440.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
526,526,2244.8440,3876.8830,5440.8570,NaN,5803.1740,6070.7954,NaN,NaN,-0.324226,2.428805,NaN,380,290_b,ffc,2022-07-30 19:40:46,5419.881,5302.6597,768.13574,554.0
527,527,2394.2498,3727.4678,5435.5913,NaN,5788.1210,5988.4960,NaN,NaN,-0.362300,2.432683,NaN,380,290_b,ffc,2022-07-30 19:40:47,5419.881,5302.6597,685.83640,532.0
528,528,2543.9016,3577.8076,5430.3340,NaN,5799.9890,5954.1367,NaN,NaN,-0.264212,2.447625,NaN,380,290_b,ffc,2022-07-30 19:40:48,5419.881,5302.6597,651.47705,529.0
529,529,2693.5527,3428.1482,5425.0850,NaN,5861.5600,5984.3170,NaN,NaN,-0.153945,2.460336,NaN,387,290_b,ffc,2022-07-30 19:40:49,5419.881,5302.6597,681.65720,555.0


In [4]:
def clean_up_fc_df(fc_df, name):
    """
    tidy up the df - remove not useful columns, rename columns to include ffc or rfc, and calculate distance of cloud top from plane height

    args:
        fc_df: dataframe of cloud top heights
        name: 'ffc' or 'rfc' - which camera the data is from
    returns:
        df_clean: cleaned up dataframe with only relevant columns and renamed columns
    """

    if name != 'ffc' and name != 'rfc':
        raise ValueError("name must be either 'ffc' or 'rfc'")

    df_main = fc_df[['d1', 'd2', 'plane_height', 'pass_number', 'timestamp']]

    # cloud top cant be below plane height - remove these values
    bad = (df_main['d1'] < df_main['plane_height']) & (df_main['d2'] < df_main['plane_height'])
    df_clean = df_main[~bad]

    # this is calculating the distance of approximated cloud top height from plane height
    # i.e. how far below cloud top
    df_clean['d1_distance'] = df_clean['d1'] - df_clean['plane_height']
    df_clean['d2_distance'] = df_clean['d2'] - df_clean['plane_height']

    cols = ['d1_distance', 'd2_distance']
    df_clean[cols] = df_clean[cols].where(df_clean[cols] >= 0)

    # rename columns to include ffc or rfc
    df_clean = df_clean.rename(columns={'d1_distance': f'{name}_d1_distance', 'd2_distance': f'{name}_d2_distance', 'timestamp': f'{name}_timestamp'})

    # keep only the relevant columns
    df_clean = df_clean[[f'{name}_d1_distance', f'{name}_d2_distance', 'pass_number', f'{name}_timestamp']]
    return df_clean

# ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~
def group_timestamps(df_clean, name):
    """
    group timestamps that are within 1 minute of each other and replace them with the earliest timestamp in the group

    args:
        df_clean: cleaned up dataframe with only relevant columns and renamed columns
        name: 'ffc' or 'rfc' - which camera the data is from
    returns:
        df: dataframe with grouped timestamps
    """
    df = df_clean.copy()
    df[f'{name}_timestamp'] = pd.to_datetime(df[f'{name}_timestamp']).dt.floor('min')

    # Sort timestamps
    df = df.sort_values(f'{name}_timestamp')

    # Create groups when the gap exceeds 1 minute
    df['group'] = (df[f'{name}_timestamp'].diff() > pd.Timedelta(minutes=1)).cumsum()

    # Replace timestamps with the earliest timestamp in each group
    df[f'{name}_timestamp'] = df.groupby('group')[f'{name}_timestamp'].transform('min')

    df = df.drop(columns='group')

    return df


# ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~
def make_mean_sd(clean_df, name):
    """
    take averages from cleaned up df. using new time to group appropriately
    
    args:
        clean_df: cleaned up dataframe with only relevant columns and renamed columns
        name: 'ffc' or 'rfc' - which camera the data is from
    returns:
        average_df: dataframe with mean and std of d1 and d2 distances for each pass
    """
    if name != 'ffc' and name != 'rfc':
        raise ValueError("name must be either 'ffc' or 'rfc'")

    pass_averages = []
    #pass_numbers = clean_df['pass_number'].unique()
    time = clean_df[f'{name}_timestamp'].unique()

    for t in time:
        sub_df = clean_df[clean_df[f'{name}_timestamp'] == t]
        d1_mean = sub_df[f'{name}_d1_distance'].mean()
        d1_std = sub_df[f'{name}_d1_distance'].std()
        d2_mean = sub_df[f'{name}_d2_distance'].mean()
        d2_std = sub_df[f'{name}_d2_distance'].std()
        d1_d2 = sub_df[[f'{name}_d1_distance',f'{name}_d2_distance']].stack() # avg of both approximations
        d1_d2_mean = d1_d2.mean()
        d1_d2_std = d1_d2.std()

        values = {
            f'{name}_d1_mean': d1_mean,
            f'{name}_d1_std': d1_std,
            f'{name}_d2_mean': d2_mean,
            f'{name}_d2_std': d2_std,
            f'{name}_d1_d2_mean': d1_d2_mean,
            f'{name}_d1_d2_std': d1_d2_std,
            f'{name}_timestamp': t
        }

        pass_averages.append(values)
    average_df = pd.DataFrame(pass_averages)
    return average_df

In [ ]:
ffc_df_clean = group_timestamps(clean_up_fc_df(ffc_df, 'ffc'), 'ffc')
rfc_df_clean = group_timestamps(clean_up_fc_df(rfc_df, 'rfc'), 'rfc')

ffc_average_df = make_mean_sd(ffc_df_clean, 'ffc')
rfc_average_df = make_mean_sd(rfc_df_clean, 'rfc')


/tmp/ipykernel_2991412/2699077045.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['d1_distance'] = df_clean['d1'] - df_clean['plane_height']
/tmp/ipykernel_2991412/2699077045.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['d2_distance'] = df_clean['d2'] - df_clean['plane_height']
/tmp/ipykernel_2991412/2699077045.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the cave

In [ ]:
#Ensure both columns are datetime
ffc_average_df['ffc_timestamp'] = pd.to_datetime(ffc_average_df['ffc_timestamp'])

rfc_average_df['rfc_timestamp'] = pd.to_datetime(rfc_average_df['rfc_timestamp'])

# Sort both dataframes by timestamp
ffc_average_df = ffc_average_df.sort_values('ffc_timestamp')
rfc_average_df = rfc_average_df.sort_values('rfc_timestamp')

# Match each FFC with the next RFC within 2 minutes
matched_df = pd.merge_asof(
    ffc_average_df,
    rfc_average_df,
    left_on='ffc_timestamp',
    right_on='rfc_timestamp',
    direction='forward',
    tolerance=pd.Timedelta('2min')
)

matched_df = matched_df[['ffc_timestamp', 'rfc_timestamp'] + [col for col in matched_df.columns if col not in ['ffc_timestamp', 'rfc_timestamp']]]

## saving to csv
#matched_df.to_csv(csv_save_loc + f'{fc_date}_matched_cloud_heights.csv', index=False)


In [ ]:
matched_df_time = matched_df[['ffc_timestamp', 'rfc_timestamp']]
print(matched_df_time)



         ffc_timestamp       rfc_timestamp
0  2022-07-30 16:29:00 2022-07-30 16:30:00
1  2022-07-30 16:38:00                 NaT
2  2022-07-30 16:42:00 2022-07-30 16:43:00
3  2022-07-30 16:50:00 2022-07-30 16:51:00
4  2022-07-30 16:54:00 2022-07-30 16:56:00
5  2022-07-30 16:59:00 2022-07-30 17:01:00
6  2022-07-30 17:03:00                 NaT
7  2022-07-30 17:08:00 2022-07-30 17:10:00
8  2022-07-30 17:14:00                 NaT
9  2022-07-30 18:21:00 2022-07-30 18:23:00
10 2022-07-30 18:24:00 2022-07-30 18:25:00
11 2022-07-30 18:43:00 2022-07-30 18:44:00
12 2022-07-30 18:50:00                 NaT
13 2022-07-30 18:54:00 2022-07-30 18:56:00
14 2022-07-30 19:00:00 2022-07-30 19:00:00
15 2022-07-30 19:04:00 2022-07-30 19:06:00
16 2022-07-30 19:10:00                 NaT
17 2022-07-30 19:12:00 2022-07-30 19:13:00
18 2022-07-30 19:39:00 2022-07-30 19:40:00


In [43]:
matched_df

,ffc_timestamp,rfc_timestamp,ffc_d1_mean,ffc_d1_std,ffc_d2_mean,ffc_d2_std,ffc_d1_d2_mean,ffc_d1_d2_std,rfc_d1_mean,rfc_d1_std,rfc_d2_mean,rfc_d2_std,rfc_d1_d2_mean,rfc_d1_d2_std
0,2022-07-30 16:29:00,2022-07-30 16:30:00,168.323300,142.602359,476.836129,308.657000,384.282280,300.400488,754.545222,420.356246,389.557500,286.985165,608.550133,405.759882
1,2022-07-30 16:38:00,NaT,721.284200,454.317481,10035.349344,728.617120,6453.016596,4663.381487,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-07-30 16:42:00,2022-07-30 16:43:00,466.529150,261.663415,13775.612717,959.071522,11022.009221,5553.251740,NaN,NaN,7181.265326,1588.566554,7181.265326,1588.566554
3,2022-07-30 16:50:00,2022-07-30 16:51:00,NaN,NaN,1365.624029,794.617587,1365.624029,794.617587,1488.003415,877.967015,NaN,NaN,1488.003415,877.967015
4,2022-07-30 16:54:00,2022-07-30 16:56:00,NaN,NaN,3999.482504,1138.996056,3999.482504,1138.996056,3288.699359,1250.926943,NaN,NaN,3288.699359,1250.926943
5,2022-07-30 16:59:00,2022-07-30 17:01:00,NaN,NaN,4626.785077,1250.087895,4626.785077,1250.087895,5829.176750,499.440075,NaN,NaN,5829.176750,499.440075
6,2022-07-30 17:03:00,NaT,72.938000,NaN,11074.403100,458.313487,10074.269909,3345.441155,NaN,NaN,NaN,NaN,NaN,NaN
7,2022-07-30 17:08:00,2022-07-30 17:10:00,NaN,NaN,7480.065838,1300.030010,7480.065838,1300.030010,8519.724715,497.556314,NaN,NaN,8519.724715,497.556314
8,2022-07-30 17:14:00,NaT,NaN,NaN,3906.938757,619.764144,3906.938757,619.764144,NaN,NaN,NaN,NaN,NaN,NaN
9,2022-07-30 18:21:00,2022-07-30 18:23:00,803.487120,770.213885,6213.613100,758.176950,3508.550110,2941.010874,5297.808580,184.718880,182.888050,92.440847,3836.402714,2500.663375


In [44]:
pass_numbers = rfc_df['pass_number'].unique()
for pass_num in pass_numbers:
    sub_df = rfc_df[rfc_df['pass_number'] == pass_num]
    print(f"Pass number: {pass_num}")
    print(sub_df)

Pass number: 271
    Unnamed: 0          d1           d2  plane_height  pass_height  \
0            0   297.75363   297.553830     5407.1025          NaN   
1            1   446.53830   148.772130     5401.0557          NaN   
2            2   595.32490     0.066383     5394.9270          NaN   
3            3   744.58370   149.267840     5388.7686          NaN   
4            4   893.05870   297.739560     5382.6310          NaN   
5            5  1042.00660   446.685520     5376.5110          NaN   
6            6  1190.95650   595.633240     5370.4424          NaN   
7            7  1340.38060   745.056150     5364.4140          NaN   
8            8  1489.02040   893.693850     5358.4320          NaN   
9            9  1637.66190  1042.333100     5352.5176          NaN   
10          10  1786.77840  1191.448400     5346.6426          NaN   
11          11  1935.89710  1340.565700     5340.8140          NaN   
12          12  2084.70430  1489.372000     5334.9630          NaN   
13 

In [42]:
pass_numbers

array(['271', '273_a', '274', '275', '276', '277', '278', '280', '281',
       '282_a', '282_c', '284_a', '284_b', '285_a', '285_b', '286_b',
       '287_b', '287_c', '288', '290_a', '290_b'], dtype=object)

In [25]:
faam_mo_paths_dic = faam_mo_files_v2() # dictionary containing dates + paths to nc files
date_dict = faam_mo_paths_dic[date]
mo_path_64hz = date_dict['mo_non_core'] 

start_times_buf, end_times_buf = cloud_pass_time_v2(mo_path_64hz, sec_buffer=5)


/home/users/esree/miniforge3/envs/vs_code_envi/lib/python3.13/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


In [26]:
start_times_buf

DatetimeIndex(['2022-07-30 15:24:30', '2022-07-30 16:38:47',
               '2022-07-30 16:44:17', '2022-07-30 16:50:51',
               '2022-07-30 16:55:28', '2022-07-30 16:59:45',
               '2022-07-30 17:04:20', '2022-07-30 17:09:18',
               '2022-07-30 17:15:02', '2022-07-30 17:56:36',
               '2022-07-30 18:22:30', '2022-07-30 18:44:05',
               '2022-07-30 18:51:03', '2022-07-30 18:55:29',
               '2022-07-30 19:01:24', '2022-07-30 19:05:14',
               '2022-07-30 19:11:11', '2022-07-30 19:15:32',
               '2022-07-30 19:20:49', '2022-07-30 19:40:45',
               '2022-07-30 19:51:52'],
              dtype='datetime64[ns]', freq=None)

In [40]:
time = avg_df[['timestamp_ffc', 'timestamp_rfc']]

print(time)

          timestamp_ffc        timestamp_rfc
0   2022-07-30 16:29:07  2022-07-30 16:30:31
1   2022-07-30 16:42:52  2022-07-30 16:44:05
2   2022-07-30 16:50:16  2022-07-30 16:51:58
3   2022-07-30 16:54:46  2022-07-30 16:56:46
4   2022-07-30 16:59:10  2022-07-30 17:01:16
5   2022-07-30 17:03:45  2022-07-30 17:06:21
6   2022-07-30 17:08:42  2022-07-30 17:11:07
7   2022-07-30 18:21:58  2022-07-30 18:23:38
8   2022-07-30 18:24:52  2022-07-30 18:26:18
9   2022-07-30 18:43:29  2022-07-30 18:44:55
10  2022-07-30 18:54:54  2022-07-30 18:56:22
11  2022-07-30 18:55:49  2022-07-30 18:57:05
12  2022-07-30 19:00:06  2022-07-30 19:01:04
13  2022-07-30 19:00:23  2022-07-30 19:01:04
14  2022-07-30 19:05:02  2022-07-30 19:07:01
15  2022-07-30 19:12:21  2022-07-30 19:13:19
16  2022-07-30 19:12:21  2022-07-30 19:13:30
17  2022-07-30 19:39:51  2022-07-30 19:40:35
18  2022-07-30 19:40:10  2022-07-30 19:41:42


In [35]:
time = avg_df[['timestamp_ffc', 'timestamp_rfc']]

print(time)

          timestamp_ffc        timestamp_rfc
0   2022-07-30 16:29:07  2022-07-30 16:30:31
1   2022-07-30 16:42:52  2022-07-30 16:44:05
2   2022-07-30 16:50:16  2022-07-30 16:51:58
3   2022-07-30 16:54:46  2022-07-30 16:56:46
4   2022-07-30 16:59:10  2022-07-30 17:01:16
5   2022-07-30 17:03:45  2022-07-30 17:06:21
6   2022-07-30 17:08:42  2022-07-30 17:11:07
7   2022-07-30 18:21:58  2022-07-30 18:23:38
8   2022-07-30 18:24:52  2022-07-30 18:26:18
9   2022-07-30 18:43:29  2022-07-30 18:44:55
10  2022-07-30 18:54:54  2022-07-30 18:56:22
11  2022-07-30 18:55:49  2022-07-30 18:57:05
12  2022-07-30 19:00:06  2022-07-30 19:01:04
13  2022-07-30 19:00:23  2022-07-30 19:01:04
14  2022-07-30 19:05:02  2022-07-30 19:07:01
15  2022-07-30 19:12:21  2022-07-30 19:13:19
16  2022-07-30 19:12:21  2022-07-30 19:13:30
17  2022-07-30 19:39:51  2022-07-30 19:40:35
18  2022-07-30 19:40:10  2022-07-30 19:41:42


In [11]:
d1_d2.std()

np.float64(300.4004875090316)

In [86]:
rfc_df_clean

,rfc_d1_distance,rfc_d2_distance,pass_number,rfc_timestamp
34,140.2653,NaN,271,2022-07-30 16:30:23
35,294.1263,NaN,271,2022-07-30 16:30:24
36,447.4953,NaN,271,2022-07-30 16:30:25
37,601.3383,6.0508,271,2022-07-30 16:30:26
38,754.7110,159.4267,271,2022-07-30 16:30:27
...,...,...,...,...
365,4830.0950,NaN,290_b,2022-07-30 19:41:38
366,4984.7940,NaN,290_b,2022-07-30 19:41:39
367,5138.6070,NaN,290_b,2022-07-30 19:41:40
368,5293.2717,NaN,290_b,2022-07-30 19:41:41


In [68]:


camera_df = pd.merge(ffc_df_clean, rfc_df_clean, on='pass_number', how='outer')

#camera_df = pd.concat([ffc_df_clean, rfc_df_clean], ignore_index=True)

In [69]:
camera_df

,ffc_d1_distance,ffc_d2_distance,pass_number,ffc_timestamp,rfc_d1_distance,rfc_d2_distance,rfc_timestamp
0,310.896,905.131,271,2022-07-30 16:29:07,140.2653,NaN,2022-07-30 16:30:23
1,310.896,905.131,271,2022-07-30 16:29:07,294.1263,NaN,2022-07-30 16:30:24
2,310.896,905.131,271,2022-07-30 16:29:07,447.4953,NaN,2022-07-30 16:30:25
3,310.896,905.131,271,2022-07-30 16:29:07,601.3383,6.0508,2022-07-30 16:30:26
4,310.896,905.131,271,2022-07-30 16:29:07,754.7110,159.4267,2022-07-30 16:30:27
...,...,...,...,...,...,...,...
5668,NaN,23.157,290_b,2022-07-30 19:40:35,4830.0950,NaN,2022-07-30 19:41:38
5669,NaN,23.157,290_b,2022-07-30 19:40:35,4984.7940,NaN,2022-07-30 19:41:39
5670,NaN,23.157,290_b,2022-07-30 19:40:35,5138.6070,NaN,2022-07-30 19:41:40
5671,NaN,23.157,290_b,2022-07-30 19:40:35,5293.2717,NaN,2022-07-30 19:41:41


In [55]:
rfc_df_clean

,d1,d2,plane_height,pass_number,timestamp,d1_cloud_distance,d2_cloud_distance
34,5352.8813,4757.5815,5212.6160,271,2022-07-30 16:30:23,140.2653,NaN
35,5501.1733,4905.8784,5207.0470,271,2022-07-30 16:30:24,294.1263,NaN
36,5648.9880,5053.6960,5201.4927,271,2022-07-30 16:30:25,447.4953,NaN
37,5797.2870,5201.9995,5195.9487,271,2022-07-30 16:30:26,601.3383,6.0508
38,5945.1080,5349.8237,5190.3970,271,2022-07-30 16:30:27,754.7110,159.4267
...,...,...,...,...,...,...,...
365,9984.9010,3863.6200,5154.8060,290_b,2022-07-30 19:41:38,4830.0950,NaN
366,10134.1270,4012.8547,5149.3330,290_b,2022-07-30 19:41:39,4984.7940,NaN
367,10282.4640,4161.1980,5143.8570,290_b,2022-07-30 19:41:40,5138.6070,NaN
368,10431.5940,4310.3360,5138.3223,290_b,2022-07-30 19:41:41,5293.2717,NaN


In [50]:
ffc_df_clean_271['d2_cloud_distance'].mean()
#ffc_df_clean_271['d2_cloud_distance'].std()


np.float64(1825.4041076923074)

In [49]:
date = '220730'
faam_mo_paths_dic = faam_mo_files_v2() # dictionary containing dates + paths to nc files
date_dict = faam_mo_paths_dic[date]
core_phys_pth = date_dict['cloud_physics_1hz']
faam_pth = date_dict['faam_core']
faam_pth_high = date_dict['faam_core_high']
mo_path = date_dict['mo_non_core_1hz']
mo_path_64hz = date_dict['mo_non_core'] 

start_times_buf, end_times_buf = cloud_pass_time_v2(mo_path_64hz, sec_buffer=5)


/home/users/esree/miniforge3/envs/vs_code_envi/lib/python3.13/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)
